# High-Resolution (256px) Leakage-Free Pipeline — Colab (T4)

Trains a larger, 256px StyleGAN2-ADA, generates and filters synthetic mammograms, and re-runs the **same patient-level, leakage-free** evaluation to test whether a higher-fidelity generator improves the classifier. Tuned for a **free Colab T4 (16 GB)** via `configs/highres_t4.yaml`.

**Before running:**
1. `Runtime -> Change runtime type -> T4 GPU` (free). With Colab Pro (A100/L4), use `configs/highres.yaml` instead for a bigger model.
2. Push your repo to GitHub (`git push origin main`) — includes the code + `data/manifest.csv`.
3. Add three Colab **Secrets** (key icon on the left, "Notebook access" ON): `GITHUB_TOKEN` (fine-grained PAT with read access to your repo), and `KAGGLE_USERNAME` + `KAGGLE_KEY` (both from the `kaggle.json` at kaggle.com -> Settings -> Create New Token).
4. `Runtime -> Run all`. On T4 the GAN takes ~2-3 h.

**If Colab disconnects during training:** the GAN checkpoints every 20 epochs and resumes automatically — just re-run cell 5 and it continues. Everything is leakage-safe: the GAN and LPIPS filter only ever see the DEV partition; the locked TEST is untouched.

In [ ]:
# 1) Confirm the GPU (want A100 / at least V100 / L4)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2) Clone the repo (private -> uses your GITHUB_TOKEN secret) and install deps
import os
os.chdir('/content')  # never sit inside the folder we are about to delete below
from google.colab import userdata
GITHUB_USER = 'rafatokairin'
REPO        = 'TCC'
BRANCH      = 'main'
token = userdata.get('GITHUB_TOKEN')   # <- the NAME of the Colab secret (not the token itself)
url = f'https://{token}@github.com/{GITHUB_USER}/{REPO}.git'
!rm -rf /content/TCC
# Skip Git-LFS smudge: the old 128px checkpoint is not needed for the high-res run.
!GIT_LFS_SKIP_SMUDGE=1 git clone --branch {BRANCH} --depth 1 {url} /content/TCC
%cd /content/TCC
# Colab already has torch/torchvision/numpy/pandas/sklearn/scipy/pillow.
!pip install -q -e . --no-deps
!pip install -q lpips imagehash torchmetrics torch-fidelity kagglehub pyyaml
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
assert os.path.isdir('/content/TCC/scripts'), 'Clone failed — check GITHUB_TOKEN secret & repo access'
print('Repo ready at', os.getcwd())

In [ ]:
# 3) Kaggle credentials + download CBIS-DDSM (for high-res JPEGs) and MIAS
import os
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
import kagglehub
cbis_path = kagglehub.dataset_download('awsaf49/cbis-ddsm-breast-cancer-image-dataset')
mias_path = kagglehub.dataset_download('kmader/mias-mammography')
cbis_jpeg = os.path.join(cbis_path, 'jpeg')
print('CBIS jpeg:', cbis_jpeg)
print('MIAS     :', mias_path)

In [ ]:
# 4) Build the 256px image set from CBIS for the SAME patient-level manifest ids
#    (orient-left + resize; labels/patients inherited from data/manifest.csv).
%cd /content/TCC
!python scripts/build_highres_dataset.py --manifest data/manifest.csv \
    --cbis-jpeg-root "{cbis_jpeg}" --out data/dataset256 --size 256

In [ ]:
# 5) Train the high-res GAN on DEV only (patient-level).
#    T4: ~2-3 h. Checkpoints every 20 epochs and RESUMES automatically, so if
#    Colab disconnects just re-run THIS cell and it continues from the checkpoint.
%cd /content/TCC
!python scripts/01_train_gan.py --config configs/highres_t4.yaml \
    --out results_highres/gan/ckpt_dev.pth

In [ ]:
# 6) Generate + LPIPS sweep + filter (tau chosen automatically from the sweep)
%cd /content/TCC
!python scripts/02_generate_and_filter.py --config configs/highres_t4.yaml \
    --ckpt results_highres/gan/ckpt_dev.pth --out results_highres/synthetic \
    --auto-threshold --min-retention 0.05

In [ ]:
# 7) Fidelity (FID/KID), classification (patient-level TEST), external MIAS, tables
%cd /content/TCC
!python scripts/03_compute_fidelity.py --config configs/highres_t4.yaml --synthetic results_highres/synthetic --out results_highres/synthetic/fidelity.json
!python scripts/04_run_classification.py --config configs/highres_t4.yaml --synthetic results_highres/synthetic --out results_highres/classification
!python scripts/06_external_validation.py --config configs/highres_t4.yaml --dataset mias --root "{mias_path}" --synthetic results_highres/synthetic --out results_highres/external_mias
!python scripts/05_make_tables.py --summary results_highres/classification/summary.json --fidelity results_highres/synthetic/fidelity.json --external results_highres/external_mias/external_summary.json --external-name MIAS --out results_highres/generated

In [ ]:
# 8) Show the headline numbers
import json
s = json.load(open('results_highres/classification/summary.json'))
print('=== INTERNAL (patient-level, 256px) ===')
for r in sorted(s['per_ratio'], key=float):
    m = s['per_ratio'][r]; lab = 'real-only' if float(r)==0 else f'{float(r):g}:1'
    print(f"  {lab:>9}: acc={m['accuracy']['mean']:.3f} f1={m['f1']['mean']:.3f} auc={m['auc']['mean']:.3f}")
print(' Wilcoxon AUC:', {r: round(v['p_adjusted'],3) for r,v in s['stats']['wilcoxon_auc'].items()})
print(json.load(open('results_highres/synthetic/fidelity.json')))

In [ ]:
# 9) Zip results (+ generated LaTeX tables + figures) and download
%cd /content/TCC
!zip -qr /content/results_highres.zip results_highres
from google.colab import files
files.download('/content/results_highres.zip')